In [2]:
import json
from openai import OpenAI

In [3]:
client = OpenAI(
    api_key="os.environ.get("OPENAI_API_KEY")"
)

In [4]:
REPHRASE_PROMPT = """\
You rephrase questions for a surgical visual-question-answering benchmark built
on laparoscopic cholecystectomy (Cholec80) video frames. Given ONE question,
produce several SEMANTICALLY EQUIVALENT variations: same meaning, same answer,
different surface form.
 
The purpose is to test whether a model answers CONSISTENTLY regardless of how the
question is worded. Therefore the answer MUST NOT change across your variations.
 
PRESERVE EXACTLY - never alter, drop, or replace with synonyms:
- the tool / instrument name (e.g. "specimen bag", "hook", "grasper")
- the phase name (e.g. "calot triangle dissection", "clipping and cutting")
- the answer type: a yes/no question stays yes/no; a counting question stays a
  counting question. NEVER turn "is X used in Y" into "what is X doing in Y" -
  that changes the answer.
 
VARY FREELY:
- wording, grammar, active/passive voice, sentence structure, clinical phrasing,
  question framing ("Is ...", "Does ...", "During ..., is ...").
 
RULES:
- Every variation must read naturally, as a surgeon might ask.
- No two variations may be identical or differ only in punctuation.
- Do NOT add information or assumptions not in the original.
 
EXAMPLES
Original: "is specimen bag used in calot triangle dissection"
- "During calot triangle dissection, is the specimen bag used?"
- "Is the specimen bag employed in the calot triangle dissection phase?"
- "Does calot triangle dissection involve the use of the specimen bag?"
 
Original: "how many tools are operating?"
- "What is the number of tools currently in operation?"
- "How many tools are in use at this moment?"
- "Count the tools that are currently operating."
 
OUTPUT - return ONLY valid JSON, no preamble, no markdown fences:
{"variations": ["...", "...", ...]}
"""

In [5]:
def _parse_json(raw: str) -> dict:
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.strip("`")
        if raw.lower().startswith("json"):
            raw = raw[4:]
    return json.loads(raw.strip())
 

In [9]:
def rephrase_question(question: str, n: int = 5,
                      tool: str | None = None, phase: str | None = None) -> list[str]:
    """Generate `n` semantically-equivalent rephrasings of `question`.
 
    If `tool` and `phase` are given, they are (a) emphasized in the prompt and
    (b) used to drop any variation that mangled them.
    """
    anchor = ""
    if tool and phase:
        anchor = f'Keep the tool name "{tool}" and the phase name "{phase}" verbatim.\n'
 
    user_msg = (
        anchor
        + f'Original question: "{question}"\n'
        + f"Generate {n} semantically equivalent variations."
    )
 
    response = client.responses.create(
        model="gpt-5.4-mini",
        input=user_msg,
        instructions=REPHRASE_PROMPT,
    )
 
    try:
        output_text = response.output[0].content[0].text
        variations = _parse_json(output_text)["variations"]
    except (json.JSONDecodeError, KeyError, IndexError) as e:
        print(f"Parse failed for: {question!r}: {e}")
        return []
 
    # Guard against drift: if anchors were given, keep only variations that
    # still contain both the tool and the phase (case-insensitive).
    if tool and phase:
        t, p = tool.lower(), phase.lower()
        variations = [v for v in variations if t in v.lower() and p in v.lower()]
 
    return variations


In [10]:
if __name__ == "__main__":
    # tool-phase yes/no question
    print(rephrase_question(
        "is specimen bag used in calot triangle dissection",
        tool="specimen bag", phase="calot triangle dissection",
    ))
    # counting question (no tool/phase anchors)
    print(rephrase_question("how many tools are operating?"))

['Is the specimen bag used during calot triangle dissection?', 'During calot triangle dissection, is the specimen bag used?', 'Does calot triangle dissection involve use of the specimen bag?', 'Is the specimen bag employed in calot triangle dissection?', 'In calot triangle dissection, is the specimen bag being used?']
['What is the number of tools currently operating?', 'How many tools are in operation?', 'Count the tools that are operating now.', 'How many tools are currently in use?', 'What count of tools is operating?']
